<a href="https://colab.research.google.com/github/Waleed-Mairaj-Malik/Autonomous-Creative-Agency/blob/main/babyagi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install babyagi

In [ ]:
!pip install openai

In [ ]:
import os
import getpass
from openai import OpenAI

# Securely prompt for your API Key
GROK_API_KEY = getpass.getpass("Enter your Grok API Key: ")

# Initialize the client using Grok's base URL endpoint
client = OpenAI(
    api_key=GROK_API_KEY,
    base_url="https://api.xai.com/v1",
)

# We will use the standard Grok text model
MODEL ="grok-4.3"

Enter your Grok API Key: ··········


In [ ]:
import time
from collections import deque

# --- 1. THE AI AGENTS (MOCKED FOR RELIABILITY) ---

# EXECUTION AGENT: Takes a task and returns the completed answer
def execution_agent(task: str) -> str:
    print(f"\n⚡ [Executing Task]: {task}")
    time.sleep(1) # Small pause to feel realistic

    # Simple logic to give smart answers based on the task name
    if "ingredients" in task.lower():
        return "Ingredients: Lemons ($10), Sugar ($5), Cups ($5). Total Cost: $20."
    elif "location" in task.lower():
        return "Best spot identified: The main community park entrance."
    else:
        return f"Task '{task}' successfully completed!"

# TASK CREATION AGENT: Looks at what we just did, and decides what to do next
def task_creation_agent(last_result: str) -> list:
    print("🧠 [Task Creator]: Analyzing results to find next steps...")

    if "Ingredients" in last_result:
        return ["Find a high-traffic location", "Design a marketing poster"]
    return ["Launch the stand and start selling!"]

# PRIORITIZATION AGENT: Sorts the tasks (In this simple version, it just cleans up the list)
def prioritization_agent(tasks: list) -> list:
    print("📋 [Prioritizer]: Organizing the queue by urgency...")
    return [t.strip() for t in tasks if t]


# --- 2. THE MAIN LOOP ENGINE ---

# Set your grand goal and the very first starting step
OBJECTIVE = "Launch a successful lemonade stand."
INITIAL_TASK = "Analyze what ingredients are needed and estimate their costs."

# The Task Queue (Waiting list of things to do)
task_queue = deque([INITIAL_TASK])

print(f"🚀 Starting BabyAGI Loop for Objective: '{OBJECTIVE}'")
print("=" * 60)

# Run the autonomous loop for exactly 2 rounds
for round_num in range(1, 3):
    if not task_queue:
        print("🎉 All tasks completed!")
        break

    print(f"\n📋 CURRENT QUEUE: {list(task_queue)}")

    # Step A: Pull the first task out of the waiting list
    current_task = task_queue.popleft()

    # Step B: Execute the task using our agent
    result = execution_agent(current_task)
    print(f"📝 [Result]: {result}")

    # Step C: Brainstorm new tasks based on that result
    new_tasks = task_creation_agent(result)

    # Step D: Add the new tasks to our queue
    for task in new_tasks:
        task_queue.append(task)

    # Step E: Let the prioritizer re-order the remaining tasks
    ordered_tasks = prioritization_agent(list(task_queue))
    task_queue = deque(ordered_tasks)

    print("-" * 60)

print("\n🏁 Loop simulation finished beautifully!")

🚀 Starting BabyAGI Loop for Objective: 'Launch a successful lemonade stand.'

📋 CURRENT QUEUE: ['Analyze what ingredients are needed and estimate their costs.']

⚡ [Executing Task]: Analyze what ingredients are needed and estimate their costs.
📝 [Result]: Ingredients: Lemons ($10), Sugar ($5), Cups ($5). Total Cost: $20.
🧠 [Task Creator]: Analyzing results to find next steps...
📋 [Prioritizer]: Organizing the queue by urgency...
------------------------------------------------------------

📋 CURRENT QUEUE: ['Find a high-traffic location', 'Design a marketing poster']

⚡ [Executing Task]: Find a high-traffic location
📝 [Result]: Best spot identified: The main community park entrance.
🧠 [Task Creator]: Analyzing results to find next steps...
📋 [Prioritizer]: Organizing the queue by urgency...
------------------------------------------------------------

🏁 Loop simulation finished beautifully!


# BABY-AGI structure-based Multi Agents
## Creative Agency

In [ ]:
!pip install -q huggingface_hub

In [ ]:
# ==============================================================================
#  BABYAGI-LITE — MULTIMODAL CREATIVE AGENCY (3-FUNCTION ARCHITECTURE)
# ==============================================================================
#
#  PIPELINE OVERVIEW
#  ------------------
#  Task 1  ->  INTERVIEW            : ask the human for poster design specs
#  Task 2  ->  PROMPT OPTIMIZATION   : turn notes into comma-separated image tags
#  Task 3  ->  VISUAL RENDERING      : send tags to Pollinations AI, display image
#
#  The loop is powered by exactly three agent functions, as required by the
#  classic BabyAGI architecture:
#
#      execution_agent()      - does the actual work for the top task
#      task_creation_agent()  - decides the next task based on the last result
#      prioritization_agent() - cleans/reorders the pending task queue
#
#  The loop has a HARD STOP the moment the poster is rendered, so there is
#  zero risk of infinite looping.
# ==============================================================================

import time
import urllib.parse
from collections import deque
from IPython.display import display, Image as IPyImage

# ------------------------------------------------------------------------------
# GLOBAL STATE
# ------------------------------------------------------------------------------
# OBJECTIVE: the high-level goal that every task ultimately serves.
OBJECTIVE = "Design a creative poster based on the user's vision and render it."

# task_queue holds dicts like {"id": int, "name": str}. deque gives us fast
# pops from the left (current task) and appends on the right (new task).
task_queue = deque()

# task_id_counter gives every new task a unique, ever-increasing ID.
task_id_counter = 1

# historical_notes accumulates context across tasks (interview answers,
# optimized tags, etc.) so later tasks can use earlier results.
historical_notes = {}

# render_complete is the hard break flag. Once True, the main loop stops,
# no matter what else is in the queue.
render_complete = False


# ------------------------------------------------------------------------------
# HELPER: free, keyless "prompt optimizer"
# ------------------------------------------------------------------------------
# This is intentionally simple and dependency-free so it NEVER fails,
# unlike third-party inference endpoints that can rate-limit or 400/402 error.
def optimize_prompt_locally(user_notes: str) -> str:
    """Convert raw user notes into clean comma-separated image-generation tags."""
    notes = user_notes.strip().lower()

    # --- 1. SUBJECT extraction -------------------------------------------------
    # Look for simple "of a ..." / "about ..." patterns, else fall back to the
    # first half of the sentence as the subject.
    if " of " in notes:
        subject = notes.split(" of ", 1)[1]
    elif " about " in notes:
        subject = notes.split(" about ", 1)[1]
    else:
        subject = notes
    subject = subject.split(",")[0].split(".")[0].strip()
    subject_tag = f"highly detailed illustration of {subject}"

    # --- 2. BACKGROUND / LAYOUT keyword matching -------------------------------
    background_map = {
        "city":        "urban cityscape background, layered skyline",
        "forest":      "dense forest background, layered foliage depth",
        "beach":       "ocean horizon background, sandy foreground",
        "space":       "deep space background, stars and nebulae",
        "minimal":     "minimalist negative-space layout, clean grid composition",
        "abstract":    "abstract geometric background composition",
        "studio":      "professional studio backdrop, centered subject layout",
        "mountain":    "mountain range background, vast open layout",
    }
    background_tag = "balanced rule-of-thirds composition, clean poster layout"
    for keyword, mapped_tag in background_map.items():
        if keyword in notes:
            background_tag = mapped_tag
            break

    # --- 3. LIGHTING / VIBE keyword matching -----------------------------------
    lighting_map = {
        "dark":      "moody low-key lighting, dramatic shadows",
        "bright":    "bright high-key lighting, vibrant tones",
        "neon":      "neon cyberpunk lighting, glowing accents",
        "vintage":   "warm vintage film lighting, retro color grade",
        "futuristic":"sleek futuristic lighting, cool chrome highlights",
        "warm":      "warm golden-hour lighting, soft glow",
        "cold":      "cold blue-toned lighting, crisp contrast",
        "elegant":   "soft elegant studio lighting, refined color palette",
    }
    lighting_tag = "cinematic dramatic lighting, high contrast vibe"
    for keyword, mapped_tag in lighting_map.items():
        if keyword in notes:
            lighting_tag = mapped_tag
            break

    # --- 4. Assemble final comma-separated tag string --------------------------
    final_tags = f"{subject_tag}, {background_tag}, {lighting_tag}, poster art, 4k"
    return final_tags


# ------------------------------------------------------------------------------
# CORE FUNCTION 1: execution_agent
# ------------------------------------------------------------------------------
def execution_agent(objective: str, task: dict, historical_notes: dict) -> str:
    """
    Executes the single top task pulled from the queue and returns a
    plain-text result string. This result becomes input for the next
    task_creation_agent() call.
    """
    task_name = task["name"].lower()

    # --------------------------------------------------------------------
    # IMPORTANT ORDERING NOTE (Requirement #3):
    # We check for "render" / "visual" FIRST, at the very top of the
    # if/elif chain. If we checked for "prompt" or "optimi" first, a task
    # named "render using optimized prompt tags" would be wrongly caught
    # by the prompt-optimization branch instead of the rendering branch.
    # --------------------------------------------------------------------
    if "render" in task_name or "visual" in task_name:
        # ---- TASK 3: VISUAL DESIGN RENDERING -----------------------------
        global render_complete
        tags = historical_notes.get("optimized_tags")
        if not tags:
            return "ERROR: No optimized tags found. Cannot render."

        print(f"\n🎨 Rendering poster with tags:\n   {tags}\n")

        # URL-encode the tags so spaces/commas are safe inside the URL.
        encoded_tags = urllib.parse.quote(tags)
        image_url = f"https://image.pollinations.ai/prompt/{encoded_tags}"

        print(f"🔗 Requesting image from Pollinations AI:\n   {image_url}\n")
        print("⏳ Generating image (this can take 10-20 seconds on the free tier)...")
        time.sleep(2)  # small visual pause so the notebook output feels paced

        # Display the image directly inside the Colab notebook.
        display(IPyImage(url=image_url))

        # Flip the hard-stop flag — the pipeline's job is done.
        render_complete = True
        return f"Poster successfully rendered and displayed from: {image_url}"

    elif "interview" in task_name or "ask the user" in task_name:
        # ---- TASK 1: INTERVIEW THE USER ----------------------------------
        print("\n🗣️  CREATIVE INTERVIEW")
        print("Describe the poster you want (subject, mood, setting, style).")
        user_notes = input("Your creative brief: ").strip()

        if not user_notes:
            user_notes = "a poster of a lone astronaut, in space, with neon lighting"
            print(f"(No input detected — using default brief: '{user_notes}')")

        historical_notes["user_notes"] = user_notes
        return f"User provided creative brief: {user_notes}"

    elif "prompt" in task_name or "optimi" in task_name:
        # ---- TASK 2: PROMPT ENGINEERING / OPTIMIZATION -------------------
        user_notes = historical_notes.get("user_notes")
        if not user_notes:
            return "ERROR: No user notes found to optimize."

        print("\n🛠️  Optimizing creative brief into image-generation tags...")
        optimized_tags = optimize_prompt_locally(user_notes)
        historical_notes["optimized_tags"] = optimized_tags

        print(f"✅ Optimized tags: {optimized_tags}")
        return f"Generated comma-separated tags: {optimized_tags}"

    else:
        # ---- FALLBACK: unrecognized task type ----------------------------
        return f"No matching execution logic for task: '{task['name']}'"


# ------------------------------------------------------------------------------
# CORE FUNCTION 2: task_creation_agent
# ------------------------------------------------------------------------------
def task_creation_agent(objective: str, last_result: str, last_task: dict) -> list:
    """
    Looks at what was just done (last_result / last_task) and returns a list
    of new task dicts (here, always exactly one) representing the absolute
    next step in the fixed creative pipeline.

    The pipeline is strictly sequential and known in advance:
        interview -> prompt optimization -> visual rendering -> STOP
    """
    last_name = last_task["name"].lower()

    if "interview" in last_name:
        next_task_name = "Optimize user notes into comma-separated prompt tags"
    elif "prompt" in last_name or "optimi" in last_name:
        next_task_name = "Render using optimized prompt tags via image API"
    else:
        # Render step (or anything else) has no successor — pipeline is done.
        next_task_name = None

    if next_task_name is None:
        return []  # No more tasks to create; this ends the pipeline.

    global task_id_counter
    task_id_counter += 1
    new_task = {"id": task_id_counter, "name": next_task_name}
    return [new_task]


# ------------------------------------------------------------------------------
# CORE FUNCTION 3: prioritization_agent
# ------------------------------------------------------------------------------
def prioritization_agent(task_list: deque) -> deque:
    """
    Cleans and reorders the waiting task list. In this fixed 3-step pipeline
    there is naturally only ever one pending task at a time, but this
    function still performs real cleanup work:
      - removes empty/duplicate task names
      - keeps tasks sorted by their original ID (FIFO order)
    This keeps the architecture faithful to BabyAGI even though our
    pipeline is linear rather than branching.
    """
    seen_names = set()
    cleaned_tasks = []

    for task in task_list:
        name = task["name"].strip()
        if name and name.lower() not in seen_names:
            seen_names.add(name.lower())
            cleaned_tasks.append(task)

    # Sort by ID to preserve strict creation order (FIFO).
    cleaned_tasks.sort(key=lambda t: t["id"])
    return deque(cleaned_tasks)


# ==============================================================================
# MAIN AUTONOMOUS LOOP
# ==============================================================================
print("=" * 70)
print("🚀 STARTING BABYAGI CREATIVE AGENCY PIPELINE")
print(f"🎯 OBJECTIVE: {OBJECTIVE}")
print("=" * 70)

# Seed the queue with Task 1 — the interview.
task_queue.append({"id": task_id_counter, "name": "Interview the user for poster design specs"})

last_result = ""
last_task = {"id": 0, "name": "INIT"}

# HARD BREAK CONDITION: the loop runs only while there are tasks AND the
# poster hasn't been rendered yet. This guarantees no infinite looping,
# since our task_creation_agent also stops producing tasks after rendering.
while task_queue and not render_complete:

    # 1) Pull and announce the current top task.
    current_task = task_queue.popleft()
    print(f"\n--- 📋 TASK #{current_task['id']}: {current_task['name']} ---")

    # 2) EXECUTE the top task.
    last_result = execution_agent(OBJECTIVE, current_task, historical_notes)
    last_task = current_task
    print(f"✔️  Result: {last_result}")

    # 3) Stop immediately if rendering just completed — absolute break.
    if render_complete:
        break

    # 4) CREATE the next task based on what just happened.
    new_tasks = task_creation_agent(OBJECTIVE, last_result, last_task)
    for new_task in new_tasks:
        task_queue.append(new_task)

    # 5) PRIORITIZE / clean the queue before the next iteration.
    task_queue = prioritization_agent(task_queue)

print("\n" + "=" * 70)
print("🏁 PIPELINE COMPLETE — poster generated, loop terminated safely.")
print("=" * 70)

🚀 STARTING BABYAGI CREATIVE AGENCY PIPELINE
🎯 OBJECTIVE: Design a creative poster based on the user's vision and render it.

--- 📋 TASK #1: Interview the user for poster design specs ---

🗣️  CREATIVE INTERVIEW
Describe the poster you want (subject, mood, setting, style).
Your creative brief: a drama island character sitting on beach , 
✔️  Result: User provided creative brief: a drama island character sitting on beach ,

--- 📋 TASK #2: Optimize user notes into comma-separated prompt tags ---

🛠️  Optimizing creative brief into image-generation tags...
✅ Optimized tags: highly detailed illustration of a drama island character sitting on beach, ocean horizon background, sandy foreground, cinematic dramatic lighting, high contrast vibe, poster art, 4k
✔️  Result: Generated comma-separated tags: highly detailed illustration of a drama island character sitting on beach, ocean horizon background, sandy foreground, cinematic dramatic lighting, high contrast vibe, poster art, 4k

--- 📋 TASK #

✔️  Result: Poster successfully rendered and displayed from: https://image.pollinations.ai/prompt/highly%20detailed%20illustration%20of%20a%20drama%20island%20character%20sitting%20on%20beach%2C%20ocean%20horizon%20background%2C%20sandy%20foreground%2C%20cinematic%20dramatic%20lighting%2C%20high%20contrast%20vibe%2C%20poster%20art%2C%204k

🏁 PIPELINE COMPLETE — poster generated, loop terminated safely.
